# JN-A - Unconditional Ingestion: a permit feed -> the v4 event stream

**One job:** turn every row the source sent into an *event*, and **prove nothing vanished.**
No classification, no housing judgement, no dedup, no date-window filtering, no value-fixing. Every
judgement is deferred downstream as a **reversible label** on permanent evidence.

**Why so strict?** Every past data-loss in this project - the 568-unit drop, the 1173 Hearst
under-count, the Logan Park collapse - happened because a gate that should only have *labelled* a row
instead *deleted* it. v4 unwelds those: ingestion is unconditional, and the one place data can still
vanish (the ingest boundary) is guarded here by a conservation proof you can re-run yourself.

**Structure is DISCOVERED, never assumed.** This notebook does **not** hard-code which row is the
header or what the date columns are called. It *finds* the header by locating the row that contains
the source's key anchors, and *maps* date columns by matching their names against a date-concept
vocabulary. The only city-specific configuration is that vocabulary and the anchor terms - data, not
logic. This is what lets the same notebook ingest another city's feed: you adjust the vocabulary, not
the code. The probe figures (32,202 rows, the per-axis counts) are kept **only as expectations to
check against** - a tripwire if discovery finds something materially different - never as structure
imposed on the file.

**The "What Just Happened" sandwich.** Every code cell has a *What this cell does* note before and a
*What just happened* note after. The first declares intent; the second confronts the result. If they
disagree, that disagreement is the finding, and the notebook makes it loud.

### The key design: one permit row is a *bundle* of lifecycle events, not one event
Each row carries up to several dated lifecycle fields. We **explode** each row into one event per
*present* date, typed by which date-concept that column maps to:

| date concept | event_type_code     | phase       | completion signal |
|--------------|---------------------|-------------|-------------------|
| submittal    | `permit_submitted`  | BP_APPLIED  | no                |
| issuance     | `permit_issued`     | BP_ISSUED   | no                |
| finaled      | `permit_finaled`    | COMPLETION  | **yes**           |
| completed    | `permit_completed`  | COMPLETION  | **yes**           |

A null date emits no event (we never invent dates). This is the *only* typing JN-A does, and it is
mechanical - driven by which date column is populated, never by reading the description. Housing /
master / phantom classification is JN-C's job, on intact evidence.

### What this cell does
Declares only **intent**, never structure. Three things: (1) where the files and the database live;
(2) the **date-concept vocabulary** - the city-specific knowledge that "a column whose name contains
'finaled' denotes the completion date" - expressed as data the discovery cell consumes; (3) the
probe **expectations**, held strictly as values to *check against*, not numbers to impose. Note what
is absent: no header-row number, no hard-coded column names. Those are discovered in the next cell.

In [1]:
from pathlib import Path
import sqlite3, json, datetime as dt, re
import pandas as pd

# --- paths (the only thing you edit for a different layout) ---
REPO       = Path.home() / "berkeley-data"
SCHEMA_SQL = REPO / "schema" / "v4" / "schema_v4.sql"
DB_PATH    = REPO / "databases" / "berkeley_housing_v4.db"
FEED_DIR   = REPO / "data" / "raw" / "cpra-downloads"
FEED_GLOB  = "BP_Annual Permit Report-*.xlsx"

# --- date-concept VOCABULARY (city-specific knowledge, expressed as DATA) ---
# Each concept lists case-insensitive substrings that, if found in a column header, identify it.
# To onboard another city, you edit THIS, not the code. Order matters only for reporting.
DATE_CONCEPTS = [
    # (concept, event_type_code, is_completion_signal, [header substrings that denote it])
    ("submittal", "permit_submitted", 0, ["submittal", "submitted", "application date", "applied"]),
    ("issuance",  "permit_issued",    0, ["issuance", "issued", "issue date"]),
    ("finaled",   "permit_finaled",   1, ["finaled", "final date", "finalized"]),
    ("completed", "permit_completed", 1, ["completed", "completion", "co date", "occupancy"]),
]
# The PERMIT-KEY concept: substrings that identify the source's natural record key.
PERMIT_KEY_TERMS = ["permitnumber", "permit number", "permit #", "permit no", "record id", "record number"]
# Convenience extracts: concept -> candidate header substrings (all optional; missing => null).
EXTRACT_TERMS = {
    "address":     ["address", "site address", "location"],
    "apn":         ["apn", "parcel", "parcel number", " parcel"],
    "units":       ["unitsadded", "units added", "net units", "number of units", "units"],
    "description": ["description", "work description", "scope", "work type description"],
}

# --- probe EXPECTATIONS (checks only, never imposed). Set any to None to skip its assertion. ---
EXPECT_TOTAL_ROWS   = 32202   # total data rows; discovery FLAGS a material mismatch, does not force it
EXPECT_DEDUP_FLOOR  = 30764   # event count must EXCEED this (else something deduped)
EXPECT_BY_TYPE = {            # per-axis anchors; None-tolerant (other cities won't have these)
    "permit_submitted": 32202, "permit_issued": 31940,
    "permit_finaled":   21650, "permit_completed": 1,
}
EXPECT_TOTAL_TOLERANCE = 0    # how many rows of drift from EXPECT_TOTAL_ROWS is acceptable before we WARN
REQUEST_WINDOW = (dt.date(2018,1,1), dt.date(2025,12,31))  # for the out-of-window FLAG (never a filter)

NOW = dt.datetime.now(dt.timezone.utc).isoformat()
print("Intent config loaded (no structure assumed).")
print("  DB target :", DB_PATH)
print("  Feed glob :", FEED_DIR / FEED_GLOB)
print("  Date concepts:", [c[0] for c in DATE_CONCEPTS])
print("  Expectations are CHECKS, not impositions. EXPECT_TOTAL_ROWS =", EXPECT_TOTAL_ROWS)

Intent config loaded (no structure assumed).
  DB target : /Users/johngage/berkeley-data/databases/berkeley_housing_v4.db
  Feed glob : /Users/johngage/berkeley-data/data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx
  Date concepts: ['submittal', 'issuance', 'finaled', 'completed']
  Expectations are CHECKS, not impositions. EXPECT_TOTAL_ROWS = 32202


### What just happened
Intent is in memory: paths, a date-concept vocabulary, and probe expectations held strictly as
checks. Crucially, **no header row and no column names are asserted** - the notebook does not yet
claim to know the file's shape. The `DATE_CONCEPTS` and term lists are the *only* city-specific
knowledge, and they are data you can edit to onboard another city without touching the ingestion
logic. If a path above is wrong for your machine, fix it before continuing.

### What this cell does
**Discovers** each file's structure instead of assuming it. For every file it: scans the first rows
to find the **header row** - the first row whose cells contain the permit-key anchor *and* at least
one date-concept term; then maps the discovered headers to date-concepts, the permit key, and the
convenience extracts via the vocabulary. It **reports everything it found** for your eyes, and
**halts** if a file lacks a discoverable header or permit key (rather than proceeding on a guess).
This is the cell that makes the notebook portable: point it at a differently-shaped feed and it
adapts, or stops and tells you why it can't.

In [2]:
def _norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower()) if s is not None else ""

def discover_header_row(path, max_scan=25):
    # Find the header row: first row containing a permit-key term AND >=1 date-concept term.
    preview = pd.read_excel(path, header=None, nrows=max_scan, dtype=str)
    date_terms = [t for _,_,_,terms in DATE_CONCEPTS for t in terms]
    for i in range(len(preview)):
        cells = [_norm(v) for v in preview.iloc[i].tolist()]
        has_key  = any(any(k in c for k in PERMIT_KEY_TERMS) for c in cells)
        has_date = any(any(d in c for d in date_terms) for c in cells)
        if has_key and has_date:
            return i
    return None

def _date_score(ncol):
    # Rank a candidate column for a date-concept. Higher = better.
    # A column literally containing 'date' is the real date field; one containing
    # 'status'/'type'/'flag' is a sibling we must NOT bind to (the Issuance Date vs
    # Issuance Status collision). 'date' wins; a status-like qualifier is penalized.
    if "date" in ncol:                                   return 3   # e.g. 'issuance date'
    if any(bad in ncol for bad in ("status","type","flag","code")): return 0  # 'issuance status'
    return 1                                             # bare concept term, e.g. 'finaled'

def map_columns(columns):
    # Map header names -> roles via the vocabulary. Returns (date_map, key_col, extracts, unmatched).
    # Uses PREFERENCE SCORING, not first-match: when a concept matches several columns
    # (e.g. 'issuance' matches both 'Issuance Date' and 'Issuance Status'), pick the
    # highest-scoring unused column (the *Date one), never just the first encountered.
    cols = list(columns)
    ncols = {c: _norm(c) for c in cols}
    used = set()
    # permit key (unchanged: first match is fine, keys don't collide with status siblings)
    key_col = next((c for c in cols if any(k in ncols[c] for k in PERMIT_KEY_TERMS)), None)
    if key_col: used.add(key_col)
    # date concepts -> BEST matching column, scored
    date_map = {}  # actual_col -> (event_type_code, is_completion_signal, concept)
    for concept, etype, is_co, terms in DATE_CONCEPTS:
        candidates = [c for c in cols if c not in used and any(t in ncols[c] for t in terms)]
        if not candidates:
            continue
        # choose the highest date-score; ties broken by original column order
        best = max(candidates, key=lambda c: (_date_score(ncols[c]), -cols.index(c)))
        # guard: if the best candidate is a known non-date sibling (score 0) AND a real
        # date column for this concept exists elsewhere, that real one would have scored 3
        # and been chosen; a remaining score-0 pick means only a status-like column matched.
        date_map[best] = (etype, is_co, concept); used.add(best)
    # convenience extracts (first match is fine; these don't have the date/status collision)
    extracts = {}
    for role, terms in EXTRACT_TERMS.items():
        for c in cols:
            if c in used: continue
            if any(t in ncols[c] for t in terms):
                extracts[role] = c; used.add(c); break
    unmatched = [c for c in cols if c not in used]
    return date_map, key_col, extracts, unmatched

feed_files = sorted(FEED_DIR.glob(FEED_GLOB))
assert feed_files, f"No files matched {FEED_GLOB} in {FEED_DIR}"

discovered = {}  # path -> dict(header_row, date_map, key_col, extracts, unmatched, columns)
for f in feed_files:
    hdr = discover_header_row(f)
    assert hdr is not None, (f"HALT: could not discover a header row in {f.name} "
                             f"(no row had a permit-key term AND a date term in the first 25 rows). "
                             f"Check the file or extend PERMIT_KEY_TERMS / DATE_CONCEPTS.")
    cols = pd.read_excel(f, header=hdr, nrows=0).columns.tolist()
    date_map, key_col, extracts, unmatched = map_columns(cols)
    assert key_col is not None, f"HALT: no permit-key column discovered in {f.name}. Headers: {cols}"
    assert date_map,            f"HALT: no date columns discovered in {f.name}. Headers: {cols}"
    discovered[f] = dict(header_row=hdr, date_map=date_map, key_col=key_col,
                         extracts=extracts, unmatched=unmatched, columns=cols)
    print(f"\n=== {f.name} ===")
    print(f"  header row (0-indexed): {hdr}")
    print(f"  permit key column     : {key_col!r}")
    print(f"  date columns mapped    :")
    for col,(et,co,concept) in date_map.items():
        print(f"      {col!r:<22} -> {et:<18} (concept '{concept}', completion={co})")
    print(f"  convenience extracts   : {extracts}")
    print(f"  UNMATCHED columns (carried only in raw_payload): {unmatched}")

# cross-file consistency note (informational, not enforced - files MAY legitimately differ)
keyset = {d['key_col'] for d in discovered.values()}
print(f"\nDiscovered permit-key column(s) across files: {keyset}",
      "(consistent)" if len(keyset)==1 else "(DIFFER - verify before trusting cross-file dedup)")


=== BP_Annual Permit Report-2018-2022.xlsx ===
  header row (0-indexed): 7
  permit key column     : 'PermitNumber'
  date columns mapped    :
      'Submittal Date'       -> permit_submitted   (concept 'submittal', completion=0)
      'Issuance Date'        -> permit_issued      (concept 'issuance', completion=0)
      'Finaled Date'         -> permit_finaled     (concept 'finaled', completion=1)
      'Completed Date'       -> permit_completed   (concept 'completed', completion=1)
  convenience extracts   : {'apn': 'Parcel Number', 'units': 'NumberUnits', 'description': 'WorkDescription'}
  UNMATCHED columns (carried only in raw_payload): ['Issuance Status', 'Unnamed: 3', 'Finaled Status', 'Completed', 'StreetNumber', 'StreetName', 'Unnamed: 12', 'StreetType', 'JobValuation', 'Unnamed: 16', 'ADU', 'Detached', 'Work Type', 'OccType', 'SubType', 'UnitsAdded', 'UnitsRemoved', 'CO Required']



=== BP_Annual Permit Report-2023-2025.xlsx ===
  header row (0-indexed): 7
  permit key column     : 'PermitNumber'
  date columns mapped    :
      'Submittal Date'       -> permit_submitted   (concept 'submittal', completion=0)
      'Issuance Date'        -> permit_issued      (concept 'issuance', completion=0)
      'Finaled Date'         -> permit_finaled     (concept 'finaled', completion=1)
      'Completed Date'       -> permit_completed   (concept 'completed', completion=1)
  convenience extracts   : {'apn': 'Parcel Number', 'units': 'NumberUnits', 'description': 'WorkDescription'}
  UNMATCHED columns (carried only in raw_payload): ['Issuance Status', 'Unnamed: 3', 'Finaled Status', 'Completed', 'StreetNumber', 'StreetName', 'Unnamed: 12', 'StreetType', 'JobValuation', 'Unnamed: 16', 'ADU', 'Detached', 'Work Type', 'OccType', 'SubType', 'UnitsAdded', 'UnitsRemoved', 'CO Required']

Discovered permit-key column(s) across files: {'PermitNumber'} (consistent)


### What just happened
The notebook now knows each file's real shape **because it found it**, not because it was told.
For each file you can see the discovered header row, the permit-key column, exactly which headers
mapped to which lifecycle concept, and - importantly - the **unmatched columns**, which are carried
verbatim in `raw_payload` but not given a role (nothing is silently ignored). If any file had no
discoverable header or key, the cell would have **halted** with a specific message rather than
guessing. If the permit-key column name differs across files, that is flagged so you don't trust a
cross-file join blindly. To onboard a different city, you would edit the vocabulary in Cell 1 and
re-run this cell - the discovery does the rest.

### What this cell does
Builds a **fresh** `berkeley_housing_v4.db` from the committed schema (deleting any prior v4 build so
the notebook is idempotent). **v3 is never touched** - there is a hard guard refusing to operate on
any file not named `berkeley_housing_v4.db`. Confirms 27 tables and clean foreign-key integrity.

In [3]:
assert DB_PATH.name == "berkeley_housing_v4.db", "Safety: refusing to build over a non-v4 file."
if DB_PATH.exists():
    DB_PATH.unlink(); print("Removed prior v4 db (fresh rebuild).")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON;")
con.executescript(SCHEMA_SQL.read_text())
con.commit()
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
fk = con.execute("PRAGMA foreign_key_check").fetchall()
print(f"Built {DB_PATH.name}: {len(tables)} tables, FK integrity {'CLEAN' if not fk else fk}")
assert len(tables) == 27, f"Expected 27 tables, got {len(tables)} - schema mismatch."
assert not fk, "FK integrity problem in fresh schema."

Removed prior v4 db (fresh rebuild).
Built berkeley_housing_v4.db: 27 tables, FK integrity CLEAN


### What just happened
A clean, empty v4 database exists with all 27 tables and no FK violations. The assertions are
tripwires: a table count that isn't 27, or any FK problem, means the schema on disk differs from
expectation and we stop before ingesting into a malformed database. v3 is untouched.

### What this cell does
Seeds the **city-neutral** lifecycle vocabulary - phases and the four `permit_*` event types - derived
from the same `DATE_CONCEPTS` the discovery used, so the database vocabulary and the discovery logic
can never drift apart. Another city's adapter maps its events into these same abstract codes.

In [4]:
phases = [
    ("CONCEPT",10,"existing","Pre-application concept/design (usually unsourced)"),
    ("ENTITLEMENT_APPLIED",20,"existing","Entitlement application submitted"),
    ("ENTITLEMENT_APPROVED",30,"existing","Land-use approval granted"),
    ("BP_APPLIED",40,"existing","Building permit submitted"),
    ("BP_ISSUED",50,"existing","Building permit issued; construction may begin"),
    ("CONSTRUCTION",60,"existing","Construction-phase inspections"),
    ("COMPLETION",70,"existing","Completion / CO / finaled"),
    ("TENURE",80,"existing","Sale / lease / occupancy (usually unsourced)"),
]
con.executemany("INSERT INTO lifecycle_phases VALUES (?,?,?,?)", phases)
# event types are derived from DATE_CONCEPTS (single source of truth shared with discovery)
phase_for = {"permit_submitted":"BP_APPLIED","permit_issued":"BP_ISSUED",
             "permit_finaled":"COMPLETION","permit_completed":"COMPLETION"}
etypes = [(et, phase_for[et], is_co, f"{concept} date") for concept,et,is_co,_ in DATE_CONCEPTS]
con.executemany("INSERT INTO event_types VALUES (?,?,?,?)", etypes)
con.commit()
print(f"Seeded {len(phases)} phases, {len(etypes)} event types (derived from DATE_CONCEPTS):")
for e in etypes: print("  ", e[0], "->", e[1], "completion=" + str(e[2]))

Seeded 8 phases, 4 event types (derived from DATE_CONCEPTS):
   permit_submitted -> BP_APPLIED completion=0
   permit_issued -> BP_ISSUED completion=0
   permit_finaled -> COMPLETION completion=1
   permit_completed -> COMPLETION completion=1


### What just happened
The vocabulary is seeded, and the event types were generated *from the same `DATE_CONCEPTS`* the
discovery cell used - so there is one source of truth, not two that could disagree. The two
completion-signal types (`permit_finaled`, `permit_completed`) are what later notebooks fold to count
completions; keeping them distinct from issuance is what lets Table A2's BP section and CO section be
two separate folds.

### What this cell does
Establishes the **conservation baseline**: reads each file using its *discovered* header row, counts
data rows per file (the denominator the final proof checks against), registers each file in `sources`
with a checksum, and opens an `ingestion_runs` row. The total is compared to the probe expectation as
a **check** - a material mismatch WARNS (it may mean the source changed), it does not override the
real count.

In [5]:
import hashlib
frames, per_file, source_id_for = [], [], {}
for f in feed_files:
    d = discovered[f]
    df = pd.read_excel(f, header=d["header_row"], dtype=str)  # dtype=str: verbatim; parse later
    df["_source_file"] = f.name
    frames.append(df); per_file.append((f.name, len(df)))
    chk = hashlib.sha256(f.read_bytes()).hexdigest()[:16]
    con.execute("INSERT INTO sources (source_kind,city,locator,retrieved_at,checksum,notes) "
                "VALUES ('cpra_permit_feed','Berkeley',?,?,?,?)",
                (str(f), NOW, chk, "permit feed; structure auto-discovered"))
    source_id_for[f.name] = con.execute("SELECT MAX(source_id) FROM sources").fetchone()[0]

raw = pd.concat(frames, ignore_index=True)
rows_in_source = len(raw)
print("Per-file data-row counts (discovered header; the conservation denominator):")
for name,n in per_file: print(f"  {name:<46} {n:>7,}")
print(f"  {'TOTAL':<46} {rows_in_source:>7,}")
if EXPECT_TOTAL_ROWS is not None:
    drift = abs(rows_in_source - EXPECT_TOTAL_ROWS)
    status = "OK" if drift <= EXPECT_TOTAL_TOLERANCE else f"WARN (drift {drift:,} vs probe {EXPECT_TOTAL_ROWS:,})"
    print(f"  vs probe expectation: {status}")

con.execute("INSERT INTO ingestion_runs (source_id,started_at,rows_in_source,rows_ingested,"
            "rows_rejected,conserved) VALUES (?,?,?,0,0,0)",
            (list(source_id_for.values())[0], NOW, rows_in_source))
RUN_ID = con.execute("SELECT MAX(run_id) FROM ingestion_runs").fetchone()[0]
con.commit()
print(f"\nIngestion run #{RUN_ID} opened. rows_in_source = {rows_in_source:,}")

Per-file data-row counts (discovered header; the conservation denominator):
  BP_Annual Permit Report-2018-2022.xlsx          18,053
  BP_Annual Permit Report-2023-2025.xlsx          14,149
  TOTAL                                           32,202
  vs probe expectation: OK

Ingestion run #1 opened. rows_in_source = 32,202


### What just happened
We have the denominator: per-file and total data-row counts, read with each file's *discovered*
header, captured before any transformation. The comparison to the probe's 32,202 is a **check** - if
it WARNs, the source likely changed since the probe and you should look before proceeding, but the
real count (not the probe) is what conservation uses. Everything was read as strings so values are
preserved verbatim, including any cumulative-restatement quirks we must not "fix" here.

### What this cell does
The heart of JN-A. For **every** source row, for **every** discovered date column that is *populated*,
emit one event - typed by the concept that column mapped to. A row with submittal + issuance + finaled
becomes three events sharing `source_record_key`. The full original row is preserved verbatim in
`raw_payload`. Dates are parsed tolerantly with honest precision. **No row filtered, no date invented,
no dedup.** Row accounting (represented vs no-event) is kept strictly separate from field-level
unparseable counts, so the conservation identity stays clean.

In [6]:
def parse_date(val):
    "Tolerant parse to (iso_date_or_None, precision)."
    if val is None or isinstance(val, float) or str(val).strip() in ("", "nan", "NaT", "None"):
        return None, "unknown"
    s = str(val).strip()
    for fmt in ("%Y-%m-%d %H:%M:%S","%Y-%m-%d","%m/%d/%Y","%m/%d/%y","%Y/%m/%d"):
        try: return dt.datetime.strptime(s, fmt).date().isoformat(), "day"
        except ValueError: continue
    try: return pd.to_datetime(s, errors="raise").date().isoformat(), "day"
    except Exception: return None, "unparseable"

def cell(row, colname):
    if not colname or colname not in row.index: return None
    v = row.get(colname)
    return None if pd.isna(v) else v

event_rows = []
per_axis = {et: 0 for _,et,_,_ in DATE_CONCEPTS}
rows_with_event = rows_no_event = rejected_fields = 0
rejected_detail = []
in_win = out_win = 0
W0, W1 = REQUEST_WINDOW

# per-file discovered maps (each row knows its file, so we use that file's mapping)
by_file = {name: discovered[f] for f in feed_files for name in [f.name]}

for _, row in raw.iterrows():
    src_name = row["_source_file"]
    dmap = by_file[src_name]["date_map"]
    ex   = by_file[src_name]["extracts"]
    permit = cell(row, by_file[src_name]["key_col"])
    permit = None if permit is None else str(permit).strip()
    payload = json.dumps({k:(None if pd.isna(v) else v) for k,v in row.items()}, ensure_ascii=False)
    sid = source_id_for[src_name]
    emitted = 0
    for col,(etype,is_co,concept) in dmap.items():
        iso, prec = parse_date(cell(row, col))
        if iso is None and prec == "unknown":     # genuinely empty -> no event (never invent)
            continue
        if iso is None and prec == "unparseable": # present but unreadable -> log as data-quality, keep going
            rejected_detail.append({"permit":permit,"file":src_name,"column":col,"raw":str(row.get(col))})
            rejected_fields += 1; continue
        d = dt.date.fromisoformat(iso)
        if W0 <= d <= W1: in_win += 1
        else: out_win += 1
        event_rows.append((etype, iso, prec, permit, int(sid), RUN_ID, payload,
                           cell(row, ex.get("address")), cell(row, ex.get("apn")),
                           cell(row, ex.get("description")),
                           None if cell(row, ex.get("units")) is None else str(cell(row, ex.get("units"))),
                           NOW))
        per_axis[etype] += 1; emitted += 1
    if emitted == 0: rows_no_event += 1
    else: rows_with_event += 1

con.executemany("INSERT INTO events (event_type_code,event_date,event_date_precision,source_record_key,"
                "source_id,ingestion_run_id,raw_payload,raw_address,raw_apn,raw_description,raw_units,"
                "created_at) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)", event_rows)
con.commit()

print(f"Emitted {len(event_rows):,} events from {rows_in_source:,} source rows.")
print(f"  rows with >=1 event : {rows_with_event:,}")
print(f"  rows with 0 events  : {rows_no_event:,}  (all dates null/unparseable - represented, NOT dropped)")
print(f"  unparseable FIELDS  : {rejected_fields:,}  (bad date cells - logged; a row with one can still")
print(f"                         contribute its valid dates and is counted with-event above)")
print("\nPer-axis event tallies", "(vs probe anchors where defined):")
for _,et,_,_ in DATE_CONCEPTS:
    anchor = (EXPECT_BY_TYPE or {}).get(et)
    note = "" if anchor is None else (f"  (probe {anchor:>6,})  {'OK' if per_axis[et]==anchor else '** MISMATCH'}")
    print(f"  {et:<18} {per_axis[et]:>7,}{note}")
print(f"\nWindow flag (recorded, never a filter): in {in_win:,} / out {out_win:,}")
if rejected_detail:
    print("\nUnparseable fields (never silent):")
    for r in rejected_detail[:20]: print("  ", r)

Emitted 85,793 events from 32,202 source rows.
  rows with >=1 event : 32,202
  rows with 0 events  : 0  (all dates null/unparseable - represented, NOT dropped)
  unparseable FIELDS  : 0  (bad date cells - logged; a row with one can still
                         contribute its valid dates and is counted with-event above)

Per-axis event tallies (vs probe anchors where defined):
  permit_submitted    32,202  (probe 32,202)  OK
  permit_issued       31,940  (probe 31,940)  OK
  permit_finaled      21,650  (probe 21,650)  OK
  permit_completed         1  (probe      1)  OK

Window flag (recorded, never a filter): in 82,939 / out 2,854


### What just happened
Every source row is exploded into its lifecycle events, typed by the *discovered* column-to-concept
mapping. Where the probe anchors are defined, each per-axis tally is checked against them - four
independent ground-truth checks, far stronger than one aggregate. A `** MISMATCH` points at exactly
which date axis lost or duplicated rows. For another city with no probe anchors, those checks simply
don't fire and the tallies are reported as facts. The window flag records how many dates fall outside
the request window (recorded, never used to filter). Unparseable cells are logged with file, column,
and raw value - never silently dropped, and never miscounted as lost rows.

### What this cell does
The **single guarded boundary**. Proves arithmetically against the database that nothing vanished,
and writes the verdict to `ingestion_runs`. CHECK 1 is pure **row** conservation (every source row is
represented in exactly one mutually-exclusive bucket). CHECK 2 is **event** conservation (events ==
sum of emitted date fields). CHECK 3 is the **dedup floor**. Unparseable fields are reported as data
quality, deliberately outside the row identity. A failure **halts** the notebook.

In [7]:
events_total = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]

row_lhs = rows_in_source
row_rhs = rows_with_event + rows_no_event           # mutually exclusive; fields are NOT a row bucket
row_ok  = (row_lhs == row_rhs)

axis_sum = sum(per_axis.values())
evt_ok   = (axis_sum == events_total)

floor_ok = (EXPECT_DEDUP_FLOOR is None) or (events_total > EXPECT_DEDUP_FLOOR)

# CHECK 4 - ANCHOR verification (Berkeley-specific safety net against MIS-DISCOVERY).
# Conservation proves we lost nothing of what we INGESTED; it canNOT prove we ingested the
# right columns. The per-axis anchors are the guard against a discovery mis-mapping (e.g.
# binding 'Issuance Status' instead of 'Issuance Date'). When anchors are DEFINED (known city),
# a mismatch HALTS - a zero-axis can never again pass as success. When anchors are absent
# (a new city), there is nothing to assert against, so we WARN loudly instead of halting.
anchor_defined = bool(EXPECT_BY_TYPE)
anchor_problems = []
if anchor_defined:
    for et, expected in EXPECT_BY_TYPE.items():
        got = per_axis.get(et, 0)
        if got != expected:
            anchor_problems.append((et, got, expected))
anchor_ok = (not anchor_defined) or (not anchor_problems)

conserved = 1 if (row_ok and evt_ok and floor_ok) else 0
con.execute("UPDATE ingestion_runs SET rows_ingested=?, rows_rejected=?, conserved=?, "
            "rejected_detail=? WHERE run_id=?",
            (events_total, rejected_fields, conserved,
             json.dumps(rejected_detail) if rejected_detail else None, RUN_ID))
con.commit()

print("CONSERVATION CHECKPOINT")
print("="*64)
print(f"CHECK 1  ROW   : {row_lhs:,} == {row_rhs:,}  -> {'PASS' if row_ok else 'FAIL'}")
print(f"         ({rows_with_event:,} with-event + {rows_no_event:,} no-event; each row in exactly one)")
print(f"CHECK 2  EVENT : sum(axes) {axis_sum:,} == events {events_total:,}  -> {'PASS' if evt_ok else 'FAIL'}")
print(f"CHECK 3  FLOOR : events {events_total:,} > {EXPECT_DEDUP_FLOOR}  -> {'PASS' if floor_ok else 'FAIL'}")
if anchor_defined:
    print(f"CHECK 4  ANCHOR: per-axis vs known Berkeley anchors -> {'PASS' if anchor_ok else 'FAIL'}")
    for et, got, expected in anchor_problems:
        print(f"         ** {et}: got {got:,}, expected {expected:,}  (DISCOVERY MIS-MAPPING - a date column was not ingested)")
else:
    print(f"CHECK 4  ANCHOR: no anchors defined (unknown city) -> WARN-only, cannot assert")
print(f"         data quality (not a conservation failure): {rejected_fields:,} unparseable date fields")
print("="*64)
print(f"ingestion_runs.conserved = {conserved}  ({'CONSERVED' if conserved else 'NOT CONSERVED - STOP'})")
assert conserved == 1, "CONSERVATION FAILED - do not proceed; data was lost or invented at ingest."
assert anchor_ok, ("ANCHOR CHECK FAILED - a known per-axis count does not match; discovery mis-mapped "
                   "a date column (likely bound a *Status/*Type column instead of the *Date column). "
                   "This is a discovery finding, not a data gap - fix the column mapping, do not proceed.")

CONSERVATION CHECKPOINT
CHECK 1  ROW   : 32,202 == 32,202  -> PASS
         (32,202 with-event + 0 no-event; each row in exactly one)
CHECK 2  EVENT : sum(axes) 85,793 == events 85,793  -> PASS
CHECK 3  FLOOR : events 85,793 > 30764  -> PASS
CHECK 4  ANCHOR: per-axis vs known Berkeley anchors -> PASS
         data quality (not a conservation failure): 0 unparseable date fields
ingestion_runs.conserved = 1  (CONSERVED)


### What just happened
The checks passed and `conserved` is 1. CHECK 1 proves every source row is represented in exactly one
bucket with no remainder and no double-counting. CHECK 2 proves the events table holds exactly as many
rows as there were populated, parseable date fields. CHECK 3 confirms the count exceeds the dedup
floor, catching an accidental dedup. Unparseable fields are a data-quality stat, kept out of the row
identity (a row with one bad date still contributes its good dates). The `assert` is the hard stop -
a failed checkpoint halts the pipeline rather than letting loss flow downstream.

### What this cell does
Writes a **standalone verifier** you can run yourself, independent of this notebook. Critically, it is
**genuinely independent of the discovery heuristic** - an earlier version re-used discovery and so
inherited the very bug it was meant to catch (it passed while 3 of 4 date axes had silently ingested
0 events). This version instead counts the **real date columns directly by their exact names** and
checks the per-type event counts against the **known anchors** - so a discovery mis-mapping makes it
FAIL, not falsely pass. *Verify, don't trust*, and make sure the verifier can actually catch the error.

In [8]:
# Derive the verifier's inputs from the ACTUAL discovery result of the first file:
# the header row found, and the real date column names that got correctly mapped.
# (These are concrete, not heuristic - the verifier counts these exact columns.)
_first = discovered[feed_files[0]]
HEADER_ROW_FOR_VERIFY = _first["header_row"]
# real_col -> event_type, from the (now correctly-scored) date_map
REAL_DATE_COLS_FOR_VERIFY = {col: etype for col,(etype,_co,_concept) in _first["date_map"].items()}
print("Verifier will independently count these real date columns:")
for col,et in REAL_DATE_COLS_FOR_VERIFY.items(): print(f"   {col!r} -> {et}")

verify = '''#!/usr/bin/env python3
# INDEPENDENT verification of JN-A. Read-only on v4.db.
# CRITICAL: this does NOT re-use the discovery heuristic (an earlier version did, and so it
# inherited the discovery bug and falsely passed). Instead it (1) counts the REAL date columns
# directly by exact name, and (2) checks the per-type event counts against KNOWN ANCHORS.
# If discovery mis-maps a column, this catches it because it counts the truth independently.
import sqlite3, glob
import pandas as pd
DB          = %(db)r
FEED        = sorted(glob.glob(%(glob)r))
HEADER_ROW  = %(hdr)r
DATE_COLS   = %(datecols)r          # exact real date column names -> expected event_type
ANCHORS     = %(anchors)r           # known per-axis event-count anchors

con = sqlite3.connect(DB)
events  = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
by_type = dict(con.execute("SELECT event_type_code,COUNT(*) FROM events GROUP BY event_type_code").fetchall())
run     = con.execute("SELECT rows_in_source,rows_ingested,rows_rejected,conserved FROM ingestion_runs ORDER BY run_id DESC LIMIT 1").fetchone()

# Independently count non-null cells in each REAL date column, straight from the files.
truth = {}
for f in FEED:
    df = pd.read_excel(f, header=HEADER_ROW, dtype=str)
    for col, etype in DATE_COLS.items():
        if col in df.columns:
            truth[etype] = truth.get(etype, 0) + int(df[col].notna().sum())

print("events in db                       :", events)
print("events by type                     :", by_type)
print("ingestion_runs (in/ingested/rej/conserved):", run)
print("independent date-column truth      :", truth)
print("known anchors                      :", ANCHORS)

problems = []
for etype, expected in ANCHORS.items():
    got = by_type.get(etype, 0)
    if got != expected:
        problems.append((etype, got, expected))
# also compare against independently-counted truth (catches anchor staleness too)
for etype, t in truth.items():
    got = by_type.get(etype, 0)
    if got != t:
        problems.append((etype + " (vs file truth)", got, t))

conserved_ok = (run is not None and run[3] == 1 and events == run[1])
if problems:
    print("INDEPENDENT VERDICT: FAIL - per-axis mismatch (discovery mis-mapping or anchor drift):")
    for et, got, exp in problems:
        print("   ", et, "got", got, "expected", exp)
elif not conserved_ok:
    print("INDEPENDENT VERDICT: FAIL - conservation flag or count mismatch")
else:
    print("INDEPENDENT VERDICT: PASS")
''' % dict(db=str(DB_PATH), glob=str(FEED_DIR / FEED_GLOB), hdr=HEADER_ROW_FOR_VERIFY,
           datecols=REAL_DATE_COLS_FOR_VERIFY, anchors=(EXPECT_BY_TYPE or {}))
out = REPO / "scripts" / "verify_jn_a_conservation.py"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(verify)
print("Wrote re-runnable verifier ->", out)
print("Run it yourself any time:  python3", out)

Verifier will independently count these real date columns:
   'Submittal Date' -> permit_submitted
   'Issuance Date' -> permit_issued
   'Finaled Date' -> permit_finaled
   'Completed Date' -> permit_completed
Wrote re-runnable verifier -> /Users/johngage/berkeley-data/scripts/verify_jn_a_conservation.py
Run it yourself any time:  python3 /Users/johngage/berkeley-data/scripts/verify_jn_a_conservation.py


### What just happened
A standalone verifier now lives at `scripts/verify_jn_a_conservation.py`. It re-discovers each file's
header by the same content-based method (so it carries no hard-coded structure either), recounts the
source independently, and re-checks against the database. Run it whenever you want to confirm fidelity
without this notebook in the loop - that independence is the point.

### What this cell does
Reports the descriptive facts of the ingested stream - no judgement. Event-type distribution, the
no-event and duplicate-key statistics, the window split, and one permit's exploded events showing its
preserved `raw_payload`. This is the honest portrait handed to JN-B (typing refinement) and JN-C (the
reversible classifier).

In [9]:
print("EVENT-TYPE DISTRIBUTION")
for r in con.execute("SELECT event_type_code,COUNT(*) FROM events GROUP BY event_type_code ORDER BY 2 DESC"):
    print(f"  {r[0]:<18} {r[1]:>7,}")
print("\nDUPLICATE source_record_key (cross-file overlap - REPORTED, not resolved)")
dup = con.execute("SELECT COUNT(*) FROM (SELECT source_record_key FROM events WHERE source_record_key IS NOT NULL "
                  "GROUP BY source_record_key HAVING COUNT(DISTINCT raw_payload) > 1)").fetchone()[0]
maxm = con.execute("SELECT MAX(c) FROM (SELECT COUNT(*) c FROM events WHERE source_record_key IS NOT NULL "
                   "GROUP BY source_record_key)").fetchone()[0]
print(f"  permit numbers in >1 distinct source row: {dup:,}")
print(f"  max events sharing one permit number    : {maxm}")
print(f"\nWINDOW SPLIT (recorded fact): in {in_win:,} / out {out_win:,}")
print("\nSAMPLE - one permit's exploded events (raw_payload preserved):")
sk = con.execute("SELECT source_record_key FROM events WHERE source_record_key IS NOT NULL "
                 "GROUP BY source_record_key HAVING COUNT(*) >= 2 LIMIT 1").fetchone()
if sk:
    for r in con.execute("SELECT event_id,event_type_code,event_date,event_date_precision FROM events "
                         "WHERE source_record_key=? ORDER BY event_date", (sk[0],)):
        print("  ", r)
    print(f"  (permit {sk[0]} - its events span its lifecycle, keyed by permit not address)")

EVENT-TYPE DISTRIBUTION
  permit_submitted    32,202
  permit_issued       31,940
  permit_finaled      21,650
  permit_completed         1

DUPLICATE source_record_key (cross-file overlap - REPORTED, not resolved)
  permit numbers in >1 distinct source row: 1,438
  max events sharing one permit number    : 6

WINDOW SPLIT (recorded fact): in 82,939 / out 2,854

SAMPLE - one permit's exploded events (raw_payload preserved):
   (1, 'permit_submitted', '2003-08-07', 'day')
   (2, 'permit_finaled', '2021-05-14', 'day')
  (permit B2003-03513 - its events span its lifecycle, keyed by permit not address)


### What just happened
The honest portrait of the ingested stream. The distribution mirrors the per-axis anchors; the
duplicate-key count is the cross-file overlap, preserved as multiple events to be resolved later as a
label, never dropped here. The sample shows one permit exploded into its lifecycle events, keyed by
permit number rather than address - the architectural point: these dated, sourced events sit on the
spine, and no projection has yet decided what building they belong to. That is JN-D's job, by folding
this intact evidence.

---
## JN-A complete - what exists, and what's next

**Exists:** a fresh `berkeley_housing_v4.db` whose `events` table holds every lifecycle event from
every source row, each preserving its full row verbatim, each traceable to a registered source, with
a **proven, re-runnable conservation guarantee** - and built on structure **discovered from the
files**, not assumed.

**Deliberately NOT done:** classify housing vs non-housing, identify masters, dedup the overlap,
resolve restatements, project structures. All downstream, all as reversible labels or re-runnable
projections over this permanent evidence.

**Portability:** the only city-specific knowledge is the `DATE_CONCEPTS` vocabulary and the anchor
terms in Cell 1 - data, not logic. Point the notebook at another city's feed, adjust that vocabulary,
and the discovery adapts or halts with a specific reason. That is the v4 promise: one method, many
cities, no hard-coded shapes.

**Next:** JN-B refines typing from the raw fields; JN-C applies the reversible housing/master/phantom
classifier on intact evidence. Neither can lose data; both are re-runnable over what JN-A proved
complete.

*Discipline: writes only to a fresh `berkeley_housing_v4.db`, never v3, commits nothing. Run it, read
the checkpoint, run the standalone verifier yourself, then decide what to commit.*